# CPU fixed-month/day U-Net vs residual-diffusion bias smoke test

This notebook strictly loads the completed `last.ckpt` and its saved `resolved.yaml`, selects the configured month/day once in every year from 1981 through 2000, and compares the paired deterministic U-Net baseline with the physical-space diffusion ensemble mean against ground truth.

The default 16-step CPU sampler and twenty annual dates are intended to catch catastrophic bias or boundary failures. They cannot establish climatological improvement. This post-training validation run explicitly evaluates the full correction at alpha=1; that diagnostic override must not be treated as deployment evidence.

In [ ]:
from pathlib import Path
import gc
import hashlib
import importlib.util
import json
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xarray as xr
from IPython.display import display
from torch.utils.data import DataLoader, Subset

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

PROJECT_DIR = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'SA_T2_ACCESS-CM2_static_residual_diffusion.yaml').is_file()
).resolve()
REPO_ROOT = PROJECT_DIR.parents[1]
RUN_NAME = 'SA_T2_ACCESS-CM2_static_residual_diffusion'
RUN_DIR = PROJECT_DIR / 'runs' / 'SA_T2_ACCESS-CM2_static_train' / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / 'checkpoints' / 'last.ckpt'
RESOLVED_CONFIG = RUN_DIR / 'config' / 'resolved.yaml'
RUN_MANIFEST = RUN_DIR / 'run_manifest.json'

DATA_ROOT = Path('D:/CORDEX/SA_domain')
PREDICTOR_PATH = DATA_ROOT / 'test/historical/predictors/perfect/ACCESS-CM2_1981-2000.nc'
TARGET_PATH = DATA_ROOT / 'test/historical/target/pr_tasmax_ACCESS-CM2_1981-2000.nc'
STATIC_PATH = DATA_ROOT / 'train/Emulator_hist_future/predictors/Static_fields.nc'

N_RANDOM_SAMPLES = 20
SPECIFIC_DATE = '02-14'           # month and day only, in MM-DD format
EVALUATION_YEARS = tuple(range(1981, 2001))
SAMPLE_SEED = 20260717
RESIDUAL_ALPHA = 1.0            # validate the full learned correction against the paired U-Net
ALLOW_RESIDUAL_ALPHA_OVERRIDE = True   # diagnostic validation only; never a deployment default
# None uses the checkpoint-native num_sampling_steps (256 for the active YAML).
# Setting an integer overrides the sampler step count for CPU speed — fewer steps
# than the training value degrade sample quality; use None for a faithful evaluation.
CPU_SAMPLING_STEPS = None       # set to an integer (e.g. 16) only for a quick smoke test
CPU_BATCH_SIZE = 1              # minimizes CPU RAM; stochastic replay depends on batch partitioning
CPU_THREADS = max(1, min(8, os.cpu_count() or 1))
BOOTSTRAP_DRAWS = 10_000
REQUIRE_FINAL_CHECKPOINT = False  # True only when a completed 10-epoch checkpoint is required
SAVE_RESULTS = True
OUTPUT_ROOT = PROJECT_DIR / 'evaluations' / 'cpu_fixed_month_day_1981_2000_bias_test'

for required in (RESOLVED_CONFIG, RUN_MANIFEST, PREDICTOR_PATH, TARGET_PATH, STATIC_PATH):
    if not required.is_file():
        raise FileNotFoundError(required)
if 'SA_RESIDUAL_ALPHA' in os.environ:
    raise RuntimeError('Remove SA_RESIDUAL_ALPHA before running; this notebook resolves and records alpha explicitly.')

torch.set_num_threads(CPU_THREADS)
print(f'Project: {PROJECT_DIR}')
print(f'Pinned checkpoint: {LAST_CHECKPOINT}')
print(f'CPU threads: {torch.get_num_threads()}')

## Strict `last.ckpt` loading

There is no fallback to `epoch_N.ckpt`, `best.ckpt`, another run, or the editable base YAML. A stable intermediate `last.ckpt` is accepted by default and its exact epoch/hash are recorded. Set `REQUIRE_FINAL_CHECKPOINT = True` when a completed run is required. Compatibility is validated before runtime-only CPU, alpha, and smoke-step overrides are applied.

In [ ]:
for path in (REPO_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from granitewxc.models.model import get_finetune_model_UNET
from granitewxc.utils.checkpoint_metadata import validate_checkpoint_compatibility
from granitewxc.utils.config import get_config

SCALER_KEY_PARTS = (
    'input_scalers_', 'output_scalers_',
    'static_input_scalers_', 'static_output_scalers_',
)

def sha256_path(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def stable_stat(path, pause=2.0):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f'{path} is missing. Wait until training has written at least one last.ckpt; '
            'this notebook never falls back to another checkpoint.'
        )
    first = path.stat()
    time.sleep(pause)
    second = path.stat()
    first_id = (first.st_size, first.st_mtime_ns)
    second_id = (second.st_size, second.st_mtime_ns)
    if first_id != second_id:
        raise RuntimeError(f'{path} changed while being inspected; wait for the current checkpoint write to finish.')
    return second_id

def strict_load_last():
    manifest = json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
    expected = {
        'run_name': RUN_NAME,
        'run_dir': str(RUN_DIR.resolve()),
        'checkpoint_dir': str((RUN_DIR / 'checkpoints').resolve()),
        'config_snapshot': str(RESOLVED_CONFIG.resolve()),
    }
    for key, value in expected.items():
        actual = manifest.get(key)
        mismatch = (
            actual != value
            if key == 'run_name'
            else actual is None or str(Path(actual).resolve()) != value
        )
        if mismatch:
            raise RuntimeError(f'Manifest mismatch for {key}: {actual!r} != {value!r}')

    before = stable_stat(LAST_CHECKPOINT)
    checkpoint_sha = sha256_path(LAST_CHECKPOINT)
    if stable_stat(LAST_CHECKPOINT, pause=0.0) != before:
        raise RuntimeError('last.ckpt changed while hashing; retry after the current checkpoint write finishes.')

    config_sha = sha256_path(RESOLVED_CONFIG)
    config = get_config(str(RESOLVED_CONFIG))
    checkpoint = torch.load(str(LAST_CHECKPOINT), map_location='cpu', weights_only=False, mmap=True)
    if stable_stat(LAST_CHECKPOINT, pause=0.0) != before:
        raise RuntimeError('last.ckpt changed while loading; discard this load and retry after the write finishes.')

    validate_checkpoint_compatibility(checkpoint, config)
    epoch = int(checkpoint.get('epoch', -1))
    completed = checkpoint.get('metadata', {}).get('completed_epochs')
    configured_epochs = int(config.num_epochs)
    if REQUIRE_FINAL_CHECKPOINT and (epoch + 1 != configured_epochs or int(completed or -1) != configured_epochs):
        raise RuntimeError(
            f'last.ckpt is not final: epoch={epoch}, completed_epochs={completed}, '
            f'configured num_epochs={configured_epochs}.'
        )

    config.device_target = 'cpu'
    config.scalers_device = 'cpu'
    config.dl_num_workers = 0
    config.batch_size = CPU_BATCH_SIZE
    config.data.static_path = str(STATIC_PATH.resolve())
    model = get_finetune_model_UNET(config)

    state = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model_state = model.state_dict()
    state_has_module = all(key.startswith('module.') for key in state)
    model_has_module = all(key.startswith('module.') for key in model_state)
    if model_has_module and not state_has_module:
        state = state.__class__((f'module.{key}', value) for key, value in state.items())
    elif state_has_module and not model_has_module:
        state = state.__class__((key[len('module.'):], value) for key, value in state.items())

    non_scaler_state = state.__class__(
        (key, value) for key, value in state.items()
        if not any(part in key for part in SCALER_KEY_PARTS)
    )
    expected_non_scaler = {
        key for key in model_state if not any(part in key for part in SCALER_KEY_PARTS)
    }
    missing = sorted(expected_non_scaler - set(non_scaler_state))
    unexpected = sorted(set(non_scaler_state) - expected_non_scaler)
    if missing or unexpected:
        raise RuntimeError(f'Strict checkpoint/model mismatch: missing={missing[:8]}, unexpected={unexpected[:8]}')
    strict_state = state.__class__(non_scaler_state)
    for key, value in model_state.items():
        if any(part in key for part in SCALER_KEY_PARTS):
            strict_state[key] = value
    model.load_state_dict(strict_state, strict=True)

    provenance = {
        'checkpoint_path': str(LAST_CHECKPOINT.resolve()),
        'checkpoint_sha256': checkpoint_sha,
        'checkpoint_size': before[0],
        'checkpoint_mtime_ns': before[1],
        'checkpoint_epoch': epoch,
        'checkpoint_global_step': checkpoint.get('global_step'),
        'completed_epochs': completed,
        'config_path': str(RESOLVED_CONFIG.resolve()),
        'config_sha256': config_sha,
        'config_fingerprint_sha256': checkpoint.get('metadata', {}).get('config_fingerprint_sha256'),
        'git_commit': checkpoint.get('metadata', {}).get('git_commit'),
    }
    del checkpoint, state, non_scaler_state, strict_state, model_state
    gc.collect()
    return config, model, provenance

config, model, checkpoint_provenance = strict_load_last()

def owner_with_attribute(module, attribute):
    pending, seen = [module], set()
    while pending:
        current = pending.pop()
        if id(current) in seen:
            continue
        seen.add(id(current))
        if hasattr(current, attribute):
            return current
        pending.extend(candidate for candidate in (getattr(current, 'module', None), getattr(current, '_orig_mod', None)) if candidate is not None)
    raise AttributeError(attribute)

model_owner = owner_with_attribute(model, 'diffusion_head')
head = model_owner.diffusion_head
if not bool(head.cfg.residual_diffusion):
    raise RuntimeError('last.ckpt is not a residual-diffusion model.')
if str(head.cfg.padding_mode).lower() != 'replicate':
    raise RuntimeError(f'Expected corrected replicate padding, got {head.cfg.padding_mode!r}.')
configured_residual_alpha = float(head.cfg.residual_application_scale)
if RESIDUAL_ALPHA is None:
    effective_residual_alpha = configured_residual_alpha
    residual_alpha_source = 'resolved_config'
else:
    requested_residual_alpha = float(RESIDUAL_ALPHA)
    if not np.isfinite(requested_residual_alpha) or not 0.0 <= requested_residual_alpha <= 1.0:
        raise ValueError('RESIDUAL_ALPHA must be None or a finite value in [0, 1].')
    if not ALLOW_RESIDUAL_ALPHA_OVERRIDE and not np.isclose(requested_residual_alpha, configured_residual_alpha):
        raise RuntimeError(
            f'Refusing to override saved residual_application_scale={configured_residual_alpha:g} with '
            f'RESIDUAL_ALPHA={requested_residual_alpha:g}. Set ALLOW_RESIDUAL_ALPHA_OVERRIDE=True only for a diagnostic run.'
        )
    effective_residual_alpha = requested_residual_alpha
    residual_alpha_source = (
        'notebook_diagnostic_override'
        if not np.isclose(effective_residual_alpha, configured_residual_alpha)
        else 'resolved_config_explicit'
    )
head.cfg.residual_application_scale = effective_residual_alpha
native_sampling_steps = int(head.cfg.num_sampling_steps)
if CPU_SAMPLING_STEPS is not None:
    if int(CPU_SAMPLING_STEPS) < 2:
        raise ValueError('CPU_SAMPLING_STEPS must be at least 2 or None.')
    head.cfg.num_sampling_steps = int(CPU_SAMPLING_STEPS)
effective_sampling_steps = int(head.cfg.num_sampling_steps)

checkpoint_provenance.update({
    'configured_residual_alpha': configured_residual_alpha,
    'residual_alpha': effective_residual_alpha,
    'residual_alpha_source': residual_alpha_source,
    'residual_alpha_override_unlocked': bool(ALLOW_RESIDUAL_ALPHA_OVERRIDE),
    'native_sampling_steps': native_sampling_steps,
    'effective_sampling_steps': effective_sampling_steps,
    'device': 'cpu',
})
display(pd.Series(checkpoint_provenance, name='value').to_frame())
if np.isclose(effective_residual_alpha, 0.0):
    print('Residual safety gate is CLOSED (alpha=0): diffusion cannot alter the paired U-Net baseline.')
else:
    print(f'WARNING: residual correction is active at alpha={effective_residual_alpha:g} ({residual_alpha_source}).')

In [ ]:
## Checkpoint convergence and sampling-step sanity check
#
# A critical pre-flight before running the reverse diffusion sampler.
# The epsilon-MSE score loss should be ≈1 at convergence; a value >> 1 means
# the score network outputs near-zero, causing DDIM to amplify prior noise by
# 1/alpha_T ≈ 156x for VPSDE with beta_max=20. In physical space this produces
# catastrophic corrections of ±500–1000 mm/day for precipitation.

from utils.diffusion_diagnostics import checkpoint_convergence_check

# Pull training residual statistics and recent loss history from the checkpoint.
residual_stats = {
    'count': head.residual_stats_count.cpu().numpy(),
    'rms': head.residual_training_stats()['rms'].cpu().numpy(),
}
ckpt_raw = torch.load(str(LAST_CHECKPOINT), map_location='cpu', weights_only=False, mmap=True)
train_loss_history = ckpt_raw.get('metadata', {}).get('train_loss_history', [])
del ckpt_raw

convergence_result = checkpoint_convergence_check(
    residual_stats,
    score_loss_history=train_loss_history,
    expected_score_loss_threshold=5.0,
    expected_rms_max=3.0,
)
print('\n--- CONVERGENCE CHECK ---')
print(f"converged: {convergence_result['converged']}")
print(f"score_loss_recent: {convergence_result['score_loss_recent']}")
print(f"residual_rms: {convergence_result['residual_rms']}")
print(f"residual_count: {convergence_result['residual_count']:.0f}")
if not convergence_result['converged']:
    print()
    for msg in convergence_result['warnings']:
        print(f"  *** {msg} ***")
    print()
    if effective_residual_alpha > 0:
        print(
            'WARNING: running with alpha>0 on an unconverged checkpoint. '
            'The diffusion correction will amplify prior noise and may '
            'catastrophically degrade the U-Net output. '
            'Set CPU_SAMPLING_STEPS=None (not 16) and ensure training has '
            f'converged (score loss << {5.0}) before drawing conclusions.'
        )

# Step-count advisory: warn if the user has overridden the native step count.
if CPU_SAMPLING_STEPS is not None:
    fraction = int(CPU_SAMPLING_STEPS) / native_sampling_steps
    if fraction < 0.5:
        print(
            f'\nWARNING: CPU_SAMPLING_STEPS={CPU_SAMPLING_STEPS} is '
            f'{fraction:.0%} of the training value ({native_sampling_steps}). '
            'Fewer steps can extrapolate incorrectly at the start of the '
            'reverse chain and exacerbate problems caused by a non-converged '
            'score network. Use CPU_SAMPLING_STEPS=None for a faithful evaluation.'
        )
print('--- END CONVERGENCE CHECK ---\n')

In [ ]:
from cordex_inference import CordexWrappedDataset, build_inference_dataset
from utils.diffusion_inference import resolve_base_seed, resolve_ensemble_size
from utils.inference_blending import resolve_boundary_mitigation_settings

base_dataset = build_inference_dataset(config, [str(PREDICTOR_PATH)], [str(TARGET_PATH)])
wrapped_dataset = CordexWrappedDataset(base_dataset)
target_positions = np.asarray(base_dataset._target_time_indices[0], dtype=np.int64)
with xr.open_dataset(TARGET_PATH) as truth_ds:
    aligned_times = truth_ds['time'].isel(time=target_positions).values
    lat = np.asarray(truth_ds[base_dataset.fine_lat_name].values, dtype=np.float64)
    lon = np.asarray(truth_ds[base_dataset.fine_lon_name].values, dtype=np.float64)
    source_units = {name: str(truth_ds[name].attrs.get('units', '')) for name in base_dataset.target_vars}

aligned_datetimes = pd.DatetimeIndex(aligned_times)
if not isinstance(N_RANDOM_SAMPLES, (int, np.integer)) or isinstance(N_RANDOM_SAMPLES, bool):
    raise TypeError('N_RANDOM_SAMPLES must be an integer.')
if N_RANDOM_SAMPLES != len(EVALUATION_YEARS) or EVALUATION_YEARS != tuple(range(1981, 2001)):
    raise ValueError('This test requires N_RANDOM_SAMPLES=20 and EVALUATION_YEARS=1981..2000.')
if not isinstance(SPECIFIC_DATE, str) or len(SPECIFIC_DATE) != 5 or SPECIFIC_DATE[2] != '-':
    raise ValueError("SPECIFIC_DATE must contain month and day only in MM-DD format, e.g. '02-14'.")
try:
    month, day = (int(part) for part in SPECIFIC_DATE.split('-'))
    pd.Timestamp(year=2000, month=month, day=day)  # leap year permits validation of 02-29
except (TypeError, ValueError) as exc:
    raise ValueError(f'Invalid SPECIFIC_DATE={SPECIFIC_DATE!r}; expected MM-DD.') from exc

available_years = np.unique(aligned_datetimes.year)
missing_years = sorted(set(EVALUATION_YEARS) - set(available_years))
if missing_years:
    raise ValueError(
        f'TARGET_PATH does not cover the required 1981-2000 period; missing years: {missing_years}. '
        f'The current aligned data span {aligned_datetimes.min().date()} to {aligned_datetimes.max().date()}. '
        'Set PREDICTOR_PATH and TARGET_PATH to matching files that contain 1981-2000.'
    )

requested_dates, selected_indices = [], []
for year in EVALUATION_YEARS:
    try:
        requested_date = pd.Timestamp(year=year, month=month, day=day)
    except ValueError as exc:
        raise ValueError(f'{SPECIFIC_DATE} is not a valid calendar date in {year}.') from exc
    matching = np.flatnonzero(aligned_datetimes.normalize() == requested_date)
    if matching.size != 1:
        raise ValueError(f'{requested_date.date()} must match exactly one aligned sample; found {matching.size}.')
    requested_dates.append(requested_date)
    selected_indices.append(int(matching[0]))
selected_indices = np.asarray(selected_indices, dtype=np.int64)
sampling_strategy = f'{SPECIFIC_DATE} in every year from 1981 through 2000'
selected_dates = pd.to_datetime(aligned_times[selected_indices])
selected_target_positions = target_positions[selected_indices]

sample_table = pd.DataFrame({
    'dataset_index': selected_indices,
    'target_index': selected_target_positions,
    'date': selected_dates,
    'year': selected_dates.year,
    'requested_month_day': SPECIFIC_DATE,
})
display(sample_table)
print('Sampling strategy:', sampling_strategy)

loader = DataLoader(
    Subset(wrapped_dataset, selected_indices.tolist()),
    batch_size=CPU_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)
target_vars = list(base_dataset.target_vars)
if target_vars != ['pr', 'tasmax']:
    raise RuntimeError(f'Expected [pr, tasmax], got {target_vars}.')
if source_units != {'pr': 'kg m-2 s-1', 'tasmax': 'K'}:
    raise RuntimeError(f'Unexpected ground-truth units: {source_units}.')
target_factors = torch.tensor([86400.0, 1.0], dtype=torch.float32).view(1, 2, 1, 1)

boundary_cfg = resolve_boundary_mitigation_settings(config)
if bool(boundary_cfg.enabled) and not bool(boundary_cfg.force_full_frame):
    raise RuntimeError('Independent tiled diffusion is not allowed; expected full-frame inference.')
ensemble_size = resolve_ensemble_size(config, head_type='diffusion')
base_seed = resolve_base_seed(config)
print({
    'samples': len(selected_indices),
    'ensemble_size': ensemble_size,
    'member_seeds': [base_seed + member for member in range(ensemble_size)],
    'alpha': effective_residual_alpha,
    'sampling_steps': effective_sampling_steps,
    'ground_truth_units': source_units,
})

In [ ]:
from contextlib import nullcontext
from tqdm.auto import tqdm
from utils.diffusion_inference import infer_batch_ensemble, reset_ensemble_generators
from utils.inference_blending import infer_batch_with_boundary_mitigation

device = torch.device('cpu')
model = model.to(device).eval()
reset_ensemble_generators(model)
prediction_parts, baseline_parts, truth_parts = [], [], []
started = time.perf_counter()

with torch.no_grad():
    for batch in tqdm(loader, desc=f'CPU selected-{len(selected_indices)} inference'):
        if 'x' not in batch or 'y' not in batch:
            raise KeyError('Inference batch must contain x and y.')
        truth_parts.append((batch['y'].float() * target_factors).cpu())
        device_batch = {key: value.to(device=device, dtype=torch.float32) for key, value in batch.items()}
        final, _, _ = infer_batch_ensemble(
            model=model,
            batch=device_batch,
            infer_batch=infer_batch_with_boundary_mitigation,
            boundary_cfg=boundary_cfg,
            head_type='diffusion',
            ensemble_size=ensemble_size,
            base_seed=base_seed,
            device=device,
            autocast_context=nullcontext,
            force_float32=True,
        )
        baseline_getter = getattr(owner_with_attribute(model, 'get_last_diffusion_baseline'), 'get_last_diffusion_baseline')
        direct_baseline = baseline_getter()
        if direct_baseline is None:
            raise RuntimeError('Model did not publish the paired deterministic baseline.')
        baseline_physical, _ = direct_baseline
        prediction_parts.append(final.detach().cpu().float())
        baseline_parts.append(baseline_physical.detach().cpu().float())

elapsed_seconds = time.perf_counter() - started
diffusion_members = torch.cat(prediction_parts, dim=0).numpy()
unet_baseline = torch.cat(baseline_parts, dim=0).numpy()
ground_truth = torch.cat(truth_parts, dim=0).numpy()
if diffusion_members.ndim != 5:
    raise RuntimeError(f'Expected [sample, ensemble, variable, lat, lon], got {diffusion_members.shape}.')
diffusion_mean = diffusion_members.mean(axis=1, dtype=np.float64).astype(np.float32)
if not (np.isfinite(diffusion_members).all() and np.isfinite(unet_baseline).all() and np.isfinite(ground_truth).all()):
    raise RuntimeError('Inference produced non-finite values.')
if np.min(diffusion_members[:, :, 0]) < -1e-6 or np.min(unet_baseline[:, 0]) < -1e-6:
    raise RuntimeError('Physical precipitation output contains negative values.')
print({
    'elapsed_minutes': elapsed_seconds / 60.0,
    'diffusion_members': diffusion_members.shape,
    'unet_baseline': unet_baseline.shape,
    'ground_truth': ground_truth.shape,
})

In [ ]:
area_weights = np.cos(np.deg2rad(lat))[:, None] * np.ones((1, lon.size), dtype=np.float64)

def daily_weighted_components(error, valid):
    weights = area_weights[None, :, :] * valid
    denominator = weights.sum(axis=(1, 2))
    if np.any(denominator <= 0):
        raise RuntimeError('A sample has no common finite grid cells.')
    daily_bias = (weights * error).sum(axis=(1, 2)) / denominator
    daily_mse = (weights * np.square(error)).sum(axis=(1, 2)) / denominator
    return daily_bias, daily_mse

def paired_bootstrap(base_bias, base_mse, diff_bias, diff_mse, draws, seed):
    generator = np.random.default_rng(seed)
    selection = generator.integers(0, len(base_bias), size=(draws, len(base_bias)))
    delta_abs_bias = np.abs(diff_bias[selection].mean(axis=1)) - np.abs(base_bias[selection].mean(axis=1))
    delta_rmse = np.sqrt(diff_mse[selection].mean(axis=1)) - np.sqrt(base_mse[selection].mean(axis=1))
    return np.quantile(delta_abs_bias, [0.025, 0.975]), np.quantile(delta_rmse, [0.025, 0.975])

metric_rows, daily_rows = [], []
units = {'pr': 'mm/day', 'tasmax': 'degC'}
for variable_index, variable in enumerate(target_vars):
    truth = ground_truth[:, variable_index].astype(np.float64)
    baseline = unet_baseline[:, variable_index].astype(np.float64)
    corrected = diffusion_mean[:, variable_index].astype(np.float64)
    members = diffusion_members[:, :, variable_index].astype(np.float64)
    valid = np.isfinite(truth) & np.isfinite(baseline) & np.isfinite(corrected) & np.isfinite(members).all(axis=1)
    base_daily_bias, base_daily_mse = daily_weighted_components(baseline - truth, valid)
    diff_daily_bias, diff_daily_mse = daily_weighted_components(corrected - truth, valid)
    base_bias, diff_bias = base_daily_bias.mean(), diff_daily_bias.mean()
    base_rmse, diff_rmse = np.sqrt(base_daily_mse.mean()), np.sqrt(diff_daily_mse.mean())
    bias_ci, rmse_ci = paired_bootstrap(
        base_daily_bias, base_daily_mse, diff_daily_bias, diff_daily_mse,
        BOOTSTRAP_DRAWS, SAMPLE_SEED + variable_index + 1,
    )
    required = truth - baseline
    generated = corrected - baseline
    residual_correlation = np.corrcoef(required[valid], generated[valid])[0, 1]
    daily_wins = int(np.count_nonzero(diff_daily_mse < base_daily_mse))
    metric_rows.append({
        'variable': variable,
        'unit': units[variable],
        'unet_bias': base_bias,
        'diffusion_bias': diff_bias,
        'delta_abs_bias': abs(diff_bias) - abs(base_bias),
        'delta_abs_bias_ci_low': bias_ci[0],
        'delta_abs_bias_ci_high': bias_ci[1],
        'unet_rmse': base_rmse,
        'diffusion_rmse': diff_rmse,
        'delta_rmse': diff_rmse - base_rmse,
        'delta_rmse_ci_low': rmse_ci[0],
        'delta_rmse_ci_high': rmse_ci[1],
        'daily_rmse_wins': daily_wins,
        'daily_rmse_trials': len(selected_dates),
        'generated_vs_required_residual_correlation': residual_correlation,
        'mean_ensemble_spread': members.std(axis=1, ddof=1)[valid].mean(),
    })
    for sample_index, date in enumerate(selected_dates):
        daily_rows.append({
            'date': date,
            'variable': variable,
            'unet_spatial_bias': base_daily_bias[sample_index],
            'diffusion_spatial_bias': diff_daily_bias[sample_index],
            'unet_spatial_rmse': np.sqrt(base_daily_mse[sample_index]),
            'diffusion_spatial_rmse': np.sqrt(diff_daily_mse[sample_index]),
            'delta_spatial_rmse': np.sqrt(diff_daily_mse[sample_index]) - np.sqrt(base_daily_mse[sample_index]),
        })

metrics = pd.DataFrame(metric_rows).set_index('variable')
daily_metrics = pd.DataFrame(daily_rows)
smoke_gate_passed = bool(((metrics['delta_abs_bias'] < 0) & (metrics['delta_rmse'] < 0)).all())
display(metrics.round(5))
print('Exploratory smoke gate (lower |bias| and RMSE for both variables):', smoke_gate_passed)
print('Negative deltas are improvements. Bootstrap intervals resample whole days, not grid cells.')

In [ ]:
## Diffusion vs baseline degradation gate
#
# Primary deployment criterion: the diffusion ensemble mean must not increase
# RMSE by more than 5% compared to the deterministic U-Net baseline for either
# variable. Failing this gate means the correction is harmful, not helpful.

from utils.diffusion_diagnostics import validate_diffusion_vs_baseline

# Rearrange from [T, V, H, W] for the utility function
degradation_report = validate_diffusion_vs_baseline(
    baseline_pred=unet_baseline,       # [T, V, H, W]
    diffusion_pred=diffusion_mean,     # [T, V, H, W]
    truth=ground_truth,                # [T, V, H, W]
    var_names=target_vars,
    tolerance=0.05,
    emit_warning=True,
)

print('\n--- DIFFUSION vs BASELINE GATE ---')
print(f"passed: {degradation_report['passed']}")
print('RMSE ratio (diffusion / U-Net, lower = better):')
for vname, ratio in degradation_report['rmse_ratio'].items():
    flag = '' if ratio <= 1.05 else '  <<< DEGRADED'
    print(f"  {vname}: {ratio:.4f}{flag}")
if degradation_report['warnings']:
    print()
    for msg in degradation_report['warnings']:
        print(f"  *** {msg} ***")
else:
    print('All variables within tolerance — diffusion head is not degrading the U-Net baseline.')
print('--- END DEGRADATION GATE ---\n')

In [ ]:
daily_fig, daily_axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
for axis, variable in zip(daily_axes, target_vars):
    subset = daily_metrics[daily_metrics['variable'] == variable]
    axis.axhline(0.0, color='black', linewidth=1)
    axis.scatter(subset['date'], subset['delta_spatial_rmse'], color='tab:purple')
    axis.set_title(f'{variable}: daily RMSE diffusion - U-Net')
    axis.set_ylabel(units[variable])
    axis.grid(alpha=0.25)
plt.show()

def plot_sample_mean_and_bias(variable, label, unit, sample_position=None):
    index = target_vars.index(variable)
    if sample_position is None:
        truth_mean = ground_truth[:, index].mean(axis=0, dtype=np.float64)
        unet_mean = unet_baseline[:, index].mean(axis=0, dtype=np.float64)
        diffusion_sample_mean = diffusion_mean[:, index].mean(axis=0, dtype=np.float64)
        truth_title = '(a) Ground Truth sample mean'
        comparison_label = f'{sampling_strategy} mean'
    else:
        truth_mean = ground_truth[sample_position, index].astype(np.float64)
        unet_mean = unet_baseline[sample_position, index].astype(np.float64)
        diffusion_sample_mean = diffusion_mean[sample_position, index].astype(np.float64)
        date_label = selected_dates[sample_position].strftime('%Y-%m-%d')
        truth_title = f'(a) Ground Truth: {date_label}'
        comparison_label = f'specific date {date_label}'
    if variable == 'tasmax':
        truth_display = truth_mean - 273.15
    else:
        truth_display = truth_mean
    differences = [unet_mean - truth_mean, diffusion_sample_mean - truth_mean]

    # Match the full-period comparison notebook: robust truth and shared bias limits.
    truth_min, truth_max = np.nanquantile(truth_display, [0.02, 0.98])
    bias_limit = max(float(np.nanquantile(np.abs(field), 0.98)) for field in differences)
    if np.isclose(bias_limit, 0):
        bias_limit = 1.0

    projection = ccrs.PlateCarree() if HAS_CARTOPY else None
    fig = plt.figure(figsize=(17, 6.0))
    positions = ((0.045, 0.24, 0.285, 0.70), (0.365, 0.24, 0.285, 0.70), (0.685, 0.24, 0.285, 0.70))
    axes = [fig.add_axes(position, projection=projection) if HAS_CARTOPY else fig.add_axes(position) for position in positions]
    truth_cax = fig.add_axes((0.075, 0.10, 0.225, 0.045))
    bias_cax = fig.add_axes((0.405, 0.10, 0.525, 0.045))
    geographic_kwargs = {'transform': projection} if HAS_CARTOPY else {}
    truth_plot = axes[0].pcolormesh(
        lon, lat, truth_display, shading='auto', cmap='viridis',
        vmin=float(truth_min), vmax=float(truth_max), **geographic_kwargs,
    )
    titles = (truth_title, '(b) UNet − Ground Truth', '(c) Diffusion ensemble mean − Ground Truth')
    axes[0].text(0.5, 0.975, titles[0], transform=axes[0].transAxes, ha='center', va='top', fontsize=12,
                 bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none', 'pad': 2})
    bias_plot = None
    for axis, field, title in zip(axes[1:], differences, titles[1:]):
        bias_plot = axis.pcolormesh(
            lon, lat, field, shading='auto', cmap='RdBu_r',
            vmin=-bias_limit, vmax=bias_limit, **geographic_kwargs,
        )
        axis.text(0.5, 0.975, title, transform=axis.transAxes, ha='center', va='top', fontsize=12,
                  bbox={'facecolor': 'white', 'alpha': 0.8, 'edgecolor': 'none', 'pad': 2})
    extent = [float(lon.min()), float(lon.max()), float(lat.min()), float(lat.max())]
    for axis in axes:
        if HAS_CARTOPY:
            axis.set_extent(extent, crs=projection)
            axis.coastlines(resolution='10m', linewidth=0.8)
            axis.add_feature(cfeature.BORDERS.with_scale('10m'), linewidth=0.5)
            grid = axis.gridlines(draw_labels=True, linewidth=0.35, alpha=0.5)
            grid.top_labels = False
            grid.right_labels = False
        else:
            axis.set(xlim=extent[:2], ylim=extent[2:], xlabel='longitude', ylabel='latitude')
            axis.grid(linewidth=0.35, alpha=0.5)
    fig.colorbar(truth_plot, cax=truth_cax, orientation='horizontal', label=unit)
    fig.colorbar(bias_plot, cax=bias_cax, orientation='horizontal', label=f'Difference ({unit})')
    fig.suptitle(
        f'{label}: {comparison_label} | alpha={effective_residual_alpha:g} | steps={effective_sampling_steps}', y=0.995
    )
    return fig

comparison_figures = {
    'pr': plot_sample_mean_and_bias('pr', 'Precipitation', 'mm/day'),
    'tasmax': plot_sample_mean_and_bias('tasmax', 'Daily maximum temperature', '°C'),
}
plt.show()

In [ ]:
if SAVE_RESULTS:
    result_dir = OUTPUT_ROOT / (
        f'{checkpoint_provenance["checkpoint_sha256"][:12]}_alpha{effective_residual_alpha:g}_steps{effective_sampling_steps}'
    )
    result_dir.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(result_dir / 'metrics.csv')
    daily_metrics.to_csv(result_dir / 'daily_metrics.csv', index=False)
    sample_table.to_csv(result_dir / 'selected_samples.csv', index=False)
    np.savez_compressed(
        result_dir / 'paired_predictions.npz',
        selected_indices=selected_indices,
        selected_target_positions=selected_target_positions,
        selected_dates=selected_dates.to_numpy(dtype='datetime64[ns]'),
        target_vars=np.asarray(target_vars),
        ground_truth=ground_truth,
        unet_baseline=unet_baseline,
        diffusion_members=diffusion_members,
        diffusion_mean=diffusion_mean,
        lat=lat,
        lon=lon,
    )
    manifest = {
        **checkpoint_provenance,
        'run_name': RUN_NAME,
        'predictor_path': str(PREDICTOR_PATH),
        'target_path': str(TARGET_PATH),
        'target_source_units': source_units,
        'target_canonical_units': {'pr': 'mm/day', 'tasmax': 'K'},
        'sample_seed': SAMPLE_SEED,
        'specific_month_day': SPECIFIC_DATE,
        'evaluation_years': list(EVALUATION_YEARS),
        'sampling_strategy': sampling_strategy,
        'selected_dataset_indices': selected_indices.tolist(),
        'selected_target_indices': selected_target_positions.tolist(),
        'selected_dates': [str(value) for value in selected_dates],
        'ensemble_size': ensemble_size,
        'member_seeds': [base_seed + member for member in range(ensemble_size)],
        'cpu_batch_size': CPU_BATCH_SIZE,
        'cpu_threads': CPU_THREADS,
        'elapsed_seconds': elapsed_seconds,
        'bootstrap_draws': BOOTSTRAP_DRAWS,
        'smoke_gate_passed': smoke_gate_passed,
        'interpretation': f'Exploratory {SPECIFIC_DATE} annual-sample CPU smoke test for 1981-2000; not a climatological deployment gate.',
    }
    (result_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    daily_fig.savefig(result_dir / 'daily_rmse_deltas.png', dpi=180, bbox_inches='tight')
    for variable, figure in comparison_figures.items():
        figure.savefig(result_dir / f'{variable}_sample_mean_comparison.png', dpi=200, facecolor='white')
    print(f'Saved CPU smoke-test artifacts to {result_dir}')

## Interpretation

A negative `delta_abs_bias` and `delta_rmse` means the diffusion ensemble mean improved that metric for the configured month/day across 1981–2000. The smoke gate requires both deltas to be negative for both variables. Even a pass is preliminary because twenty annual dates and a reduced-step sampler do not represent the full climatology. Confirm with a full-period, checkpoint-native evaluation; do not choose a new alpha from this test set.